# 🚀 AS-STUDIO-PRO V2.1 — Advanced Lip-Sync Studio
Run **Step 1** to setup dependencies and download models. Then run **Step 2** to launch the Gradio UI.

In [ ]:
# ==========================================
# STEP 1: SETUP DEPENDENCIES & CLEAN CHECKPOINT
# ==========================================
!apt-get update -qq && apt-get install -y -qq ffmpeg wget
!pip install -q omnivoice gradio soundfile "librosa>=0.10.0" ffmpeg-python moviepy opencv-python torch torchvision tqdm huggingface_hub

import os
import shutil
from huggingface_hub import hf_hub_download

HF_TOKEN = "" 

if not os.path.exists('/content/Wav2Lip'):
    !git clone https://github.com/Rudrabha/Wav2Lip.git

%cd /content/Wav2Lip

!mkdir -p face_detection/detection checkpoints

# Fix Librosa compatibility
audio_py_path = "/content/Wav2Lip/audio.py"
if os.path.exists(audio_py_path):
    with open(audio_py_path, "r") as f:
        content = f.read()
    old_code = "return librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,"
    new_code = "return librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels,"
    if old_code in content:
        with open(audio_py_path, "w") as f:
            f.write(content.replace(old_code, new_code))

# Fix PyTorch weights_only safety issue
inference_py_path = "/content/Wav2Lip/inference.py"
if os.path.exists(inference_py_path):
    with open(inference_py_path, "r") as f:
        inf_content = f.read()
    old_torch_load = "checkpoint = torch.load(checkpoint_path)"
    new_torch_load = "checkpoint = torch.load(checkpoint_path, weights_only=False)"
    if old_torch_load in inf_content:
        with open(inference_py_path, "w") as f:
            f.write(inf_content.replace(old_torch_load, new_torch_load))

# Download Face Detector Weights
if not os.path.exists("face_detection/detection/s3fd-619a5ba8.pth"):
    !wget -q "https://www.adrianbulat.com/downloads/python-s3fd/s3fd-619a5ba8.pth" -O "face_detection/detection/s3fd-619a5ba8.pth"

# Download Wav2Lip GAN Checkpoint safely
ckpt_path = "checkpoints/wav2lip_gan.pth"
if os.path.exists(ckpt_path) and os.path.getsize(ckpt_path) < 400 * 1024 * 1024:
    print("[!] Removing incomplete checkpoint...")
    os.remove(ckpt_path)

if not os.path.exists(ckpt_path):
    print("[+] Downloading Wav2Lip GAN model...")
    try:
        downloaded_file = hf_hub_download(repo_id="camenduru/Wav2Lip", filename="checkpoints/wav2lip_gan.pth", token=HF_TOKEN if HF_TOKEN else None)
        shutil.copy(downloaded_file, ckpt_path)
    except Exception as e:
        print(f"[!] Mirror download failed: {e}")

size_mb = os.path.getsize(ckpt_path) / (1024 * 1024) if os.path.exists(ckpt_path) else 0
print(f"\n✅ SETUP COMPLETE! Checkpoint File Size: {size_mb:.2f} MB")


In [ ]:
# ==========================================
# STEP 2: LAUNCH GRADIO WEB UI (1080p HQ)
# ==========================================
import gradio as gr
import subprocess
import os

def process_lip_sync(video_path, audio_path, pads_str, resize_factor):
    if not video_path or not audio_path:
        return None, "❌ Please upload both Video and Audio files!"
    
    output_path = "/content/Wav2Lip/results/result_voice.mp4"
    os.makedirs("/content/Wav2Lip/results", exist_ok=True)
    
    pad_args = [int(p) for p in pads_str.split()] if pads_str else [0, 10, 0, 0]
    
    cmd = [
        "python", "inference.py",
        "--checkpoint_path", "checkpoints/wav2lip_gan.pth",
        "--face", video_path,
        "--audio", audio_path,
        "--outfile", output_path,
        "--resize_factor", str(resize_factor),
        "--pads", str(pad_args[0]), str(pad_args[1]), str(pad_args[2]), str(pad_args[3])
    ]
    
    print("[+] Processing Lip-Sync Video...")
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if os.path.exists(output_path):
        return output_path, "✅ Lip-Sync Successfully Completed!"
    else:
        return None, f"❌ Error: {result.stderr}"

with gr.Blocks(title="AS-STUDIO-PRO V2.1 — Advanced Lip-Sync Studio") as demo:
    gr.Markdown("# 🚀 AS-STUDIO-PRO V2.1 — Advanced Lip-Sync Studio")
    gr.Markdown("Upload your target video and audio file to create seamless HD Lip-Sync output.")
    
    with gr.Row():
        with gr.Column():
            video_input = gr.Video(label="Input Video")
            audio_input = gr.Audio(label="Input Audio", type="filepath")
            pads_input = gr.Textbox(value="0 10 0 0", label="Padding (Top Bottom Left Right)")
            resize_input = gr.Slider(minimum=1, maximum=4, value=1, step=1, label="Resize Factor")
            btn = gr.Button("🔥 Generate Lip-Sync Video", variant="primary")
        
        with gr.Column():
            video_output = gr.Video(label="Output Video")
            status_output = gr.Textbox(label="Status Log")
            
    btn.click(
        fn=process_lip_sync,
        inputs=[video_input, audio_input, pads_input, resize_input],
        outputs=[video_output, status_output]
    )

print("🚀 Launching AS-STUDIO-PRO V2.1...")
demo.launch(share=True, debug=True)
